# Building and Loading a Software Dependency Graph into Neo4j

We will use the [deps.dev](https://deps.dev/) software dependency dataset, to build a comprehensive graph of software dependencies and load it into a Neo4j database for analysis. We will use the deps.dev public API, and will recursively fetch dependency information for a set of software packages up to a given depth.

In [1]:
# Load .env
from dotenv import load_dotenv
import os

load_dotenv()

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")
SEED_PACKAGES = [
    pkg.strip() for pkg in os.getenv("SEED_PACKAGES", "").split(",") if pkg.strip()
]
SEED_SYSTEM = os.getenv("SEED_SYSTEM", "pypi")
SEED_DEPTH = int(os.getenv("SEED_DEPTH", 2))

In [2]:
import time
import random
import requests
import pandas as pd
import urllib.parse
from tqdm.auto import tqdm
import hashlib
import json
from pathlib import Path
from functools import wraps

CACHE_DIR = Path(".cache")


def disk_cache(func):
    """Cache a function's JSON-serializable return value on disk under CACHE_DIR,
    keyed by its arguments. Repeat calls with the same arguments are served from
    disk instead of hitting the network. Only successful returns are cached;
    exceptions (e.g. 404s) propagate uncached so failed lookups can be retried.
    """

    @wraps(func)
    def wrapper(*args, **kwargs):
        CACHE_DIR.mkdir(exist_ok=True)
        key_src = json.dumps(
            [func.__name__, args, kwargs], sort_keys=True, default=str
        )
        key = hashlib.sha256(key_src.encode()).hexdigest()
        cache_file = CACHE_DIR / f"{key}.json"

        if cache_file.exists():
            try:
                return json.loads(cache_file.read_text())
            except (json.JSONDecodeError, OSError):
                pass  # corrupt/unreadable entry: fall through and refetch

        result = func(*args, **kwargs)
        try:
            cache_file.write_text(json.dumps(result))
        except (TypeError, OSError):
            pass  # non-serializable or unwritable: return without caching
        return result

    return wrapper


@disk_cache
def api_get_with_backoff(
    url: str, max_retries: int = 5, base_delay: float = 5.0
) -> dict:
    """
    Safely handles network calls with exponential backoff.
    Fails fast without retrying on strict client errors like 404.
    """
    delay = base_delay
    for attempt in range(max_retries):
        try:
            response = requests.get(url, timeout=10)

            if response.status_code == 200:
                return response.json()

            if response.status_code in [429, 500, 502, 503, 504]:
                print(
                    f"[Warning] HTTP {response.status_code}. Retrying in {delay:.2f}s... (Attempt {attempt + 1}/{max_retries})"
                )
                time.sleep(delay + random.uniform(0, 0.5))
                delay *= 2
                continue

            response.raise_for_status()

        except requests.exceptions.HTTPError as http_err:
            raise http_err

        except requests.RequestException as net_err:
            print(
                f"[Warning] Connection anomaly ({net_err}). Retrying in {delay:.2f}s... (Attempt {attempt + 1}/{max_retries})"
            )
            time.sleep(delay + random.uniform(0, 0.5))
            delay *= 2

    raise Exception(f"Max retries exceeded for URL: {url}")


def get_latest_package_dependencies_flat(
    system: str, package_name: str, max_depth: int = 3
) -> pd.DataFrame:
    """
    Fetches the pre-resolved dependency tree from deps.dev. If the absolute latest
    version hasn't been fully processed by Google yet, it automatically backtracks
    chronologically to find the most recent version with a resolved dependency graph.

    Packages that resolve cleanly but have no dependencies (leaf packages) still
    return a single node-only record so they become first-class graph nodes and are
    scanned for vulnerabilities.
    """
    system = system.upper()
    encoded_package = urllib.parse.quote(package_name, safe="")

    # 1. Fetch all package metadata and available versions
    package_url = (
        f"https://api.deps.dev/v3/systems/{system.lower()}/packages/{encoded_package}"
    )
    try:
        package_data = api_get_with_backoff(package_url)
    except Exception as e:
        print(f"[Error] Could not find package metadata for '{package_name}': {e}")
        return pd.DataFrame()

    versions = package_data.get("versions", [])
    if not versions:
        print(f"[Warning] No versions found for {package_name}")
        return pd.DataFrame()

    # 2. Sort versions chronologically (newest first) based on publishedAt timestamp
    versions_sorted = sorted(
        versions, key=lambda x: x.get("publishedAt", ""), reverse=True
    )

    # Build prioritized candidate queue: Default version first, then newest-to-oldest
    candidate_versions = []
    for v in versions:
        if v.get("isDefault"):
            candidate_versions.append(v["versionKey"]["version"])
            break

    for v in versions_sorted:
        v_str = v["versionKey"]["version"]
        if v_str not in candidate_versions:
            candidate_versions.append(v_str)

    # 3. Loop through candidates until deps.dev returns a resolved dependency graph.
    #    A resolved graph always contains at least the root node; an EMPTY `edges`
    #    list is perfectly valid (leaf packages that have no dependencies). Only the
    #    root node carrying resolution `errors` signals a genuinely unprocessed graph.
    #    Versions deps.dev has not processed at all return 404 and are handled by the
    #    except/continue below.
    deps_data = None
    resolved_version = None

    for v_str in candidate_versions:
        deps_url = f"https://api.deps.dev/v3/systems/{system.lower()}/packages/{encoded_package}/versions/{v_str}:dependencies"
        try:
            data = api_get_with_backoff(deps_url)
            candidate_nodes = data.get("nodes") or []
            root_errors = candidate_nodes[0].get("errors") if candidate_nodes else None
            if candidate_nodes and not root_errors:
                deps_data = data
                resolved_version = v_str
                break
        except Exception:
            # If a version returns 404 or fails to resolve, catch it and try the next version
            continue

    if not deps_data:
        print(
            f"[Warning] deps.dev has not resolved a dependency graph for ANY version of '{package_name}' yet."
        )
        return pd.DataFrame()

    # Log if we had to fall back from the preferred (default) version
    if resolved_version != candidate_versions[0]:
        print(
            f"--> Note: Preferred version ({candidate_versions[0]}) has no resolved graph in deps.dev. Fell back to '{resolved_version}'."
        )
    else:
        print(f"--> Using version '{resolved_version}' for '{package_name}'.")

    nodes = deps_data.get("nodes", [])
    edges = deps_data.get("edges", [])

    # 4. Robust Root Node Detection
    # Prefer the node deps.dev marks as the queried package (relation == "SELF"),
    # which is immune to PyPI name normalization (e.g. '_' vs '-'); fall back to a
    # case-insensitive name match, then to index 0.
    root_idx = None
    for idx, node in enumerate(nodes):
        if node.get("relation") == "SELF":
            root_idx = idx
            break
    if root_idx is None:
        for idx, node in enumerate(nodes):
            node_name = node.get("versionKey", {}).get("name", "")
            if node_name.lower() == package_name.lower():
                root_idx = idx
                break
    if root_idx is None:
        root_idx = 0  # Fallback to index 0 if not explicitly matched

    # 5. Local In-Memory BFS to calculate exact tree depth from the discovered root node
    node_depths = {root_idx: 0}
    queue = [root_idx]

    adj = {}
    for edge in edges:
        f = int(edge.get("fromNode", 0))
        t = int(edge.get("toNode", 0))
        if f not in adj:
            adj[f] = []
        adj[f].append(t)

    head = 0
    while head < len(queue):
        curr = queue[head]
        head += 1
        curr_depth = node_depths[curr]

        if curr_depth >= max_depth:
            continue

        for neighbor in adj.get(curr, []):
            if neighbor not in node_depths:
                node_depths[neighbor] = curr_depth + 1
                queue.append(neighbor)

    # 6. Extract all graph edges matching your depth criteria
    records = []
    for edge in edges:
        f = int(edge.get("fromNode", 0))
        t = int(edge.get("toNode", 0))

        if f in node_depths and node_depths[f] < max_depth:
            source_node = nodes[f].get("versionKey", {})
            target_node = nodes[t].get("versionKey", {})

            records.append(
                {
                    "ecosystem": system,
                    "source_package": source_node.get("name"),
                    "source_version": source_node.get("version"),
                    "target_package": target_node.get("name"),
                    "target_version": target_node.get("version"),
                    "semver_requirement": edge.get("requirement", ""),
                    "graph_depth": node_depths[f] + 1,
                }
            )

    # 7. Leaf-package handling: if the package resolved but produced no edges within
    #    max_depth, still emit a single node-only record. The null target is skipped
    #    by the relationship loader, but the source package-version makes the package
    #    a first-class graph node and feeds it into the vulnerability scan.
    if not records:
        root_node = nodes[root_idx].get("versionKey", {}) if nodes else {}
        records.append(
            {
                "ecosystem": system,
                "source_package": root_node.get("name") or package_name,
                "source_version": root_node.get("version") or resolved_version,
                "target_package": None,
                "target_version": None,
                "semver_requirement": "",
                "graph_depth": 0,
            }
        )

    return pd.DataFrame(records)


def crawl_seed_packages_flat(
    system: str, seed_packages: list, max_depth: int = 3
) -> pd.DataFrame:
    """Breadth-first crawl across the deps.dev API.

    deps.dev only returns shallow (often direct-only) dependency graphs for PyPI,
    so a single call per seed never expands transitively. This walks the graph
    itself: every newly discovered dependency package is re-queried, one BFS level
    (one dependency hop) at a time, until ``max_depth`` hops away from a seed.

    Each package is fetched at most once (deduplicated by name via ``visited``),
    and every fetch uses ``max_depth=1`` so the per-response BFS yields only that
    package's direct dependencies; depth across the wider graph comes from the
    outer level loop, not from the single API response.
    """
    all_dfs = []
    visited = set()

    # Seed packages form BFS level 0.
    current_level = []
    for pkg in seed_packages:
        if pkg and pkg not in visited:
            visited.add(pkg)
            current_level.append(pkg)

    for depth in range(max_depth + 1):
        if not current_level:
            break

        print(
            f"\n=== BFS level {depth}: crawling {len(current_level)} package(s) "
            f"(discovered total: {len(visited)}) ==="
        )

        next_level = set()
        for package in tqdm(current_level, desc=f"Level {depth}"):
            # max_depth=1 -> the response's local BFS keeps only direct edges.
            df = get_latest_package_dependencies_flat(system, package, max_depth=1)
            if df.empty:
                continue

            all_dfs.append(df)

            # Queue every newly seen dependency for the next hop. Leaf packages
            # carry a null target_package, which dropna() removes here.
            if depth < max_depth:
                for tgt in df["target_package"].dropna().unique():
                    if tgt not in visited:
                        visited.add(tgt)
                        next_level.add(tgt)

        current_level = sorted(next_level)

    if all_dfs:
        master_df = pd.concat(all_dfs, ignore_index=True)
        master_df.drop_duplicates(
            subset=["ecosystem", "source_package", "source_version", "target_package"],
            inplace=True,
        )
        print(
            f"\nCrawl complete: {len(visited)} unique packages visited, "
            f"{len(master_df)} dependency records collected."
        )
        return master_df
    return pd.DataFrame()


def get_package_version_vulnerabilities(
    system: str, package_name: str, version: str
) -> list:
    """
    Retrieves detailed security advisories (including CVE IDs, titles, and CVSS scores)
    for a specific package version.
    """
    system = system.upper()
    encoded_package = urllib.parse.quote(package_name, safe="")
    encoded_version = urllib.parse.quote(version, safe="")

    version_url = f"https://api.deps.dev/v3/systems/{system.lower()}/packages/{encoded_package}/versions/{encoded_version}"

    try:
        version_data = api_get_with_backoff(version_url)
    except Exception as e:
        # Pass silently or log if a version layout throws errors
        return []

    advisory_keys = version_data.get("advisoryKeys", [])
    vulnerabilities = []

    for adv_key in advisory_keys:
        adv_id = adv_key.get("id")
        if not adv_id:
            continue

        advisory_url = (
            f"https://api.deps.dev/v3/advisories/{urllib.parse.quote(adv_id, safe='')}"
        )
        try:
            adv_data = api_get_with_backoff(advisory_url)

            aliases = adv_data.get("aliases", [])
            title = adv_data.get("title", "")
            url = adv_data.get("url", "")
            cvss_score = adv_data.get("cvss3Score", None)
            cvss_vector = adv_data.get("cvss3Vector", "")

            # Filter or extract formal CVE tracking IDs out of the aliases array
            cve_ids = [alias for alias in aliases if alias.startswith("CVE-")]

            # Fallback: if no CVE alias exists but the advisory ID is a CVE string, adopt it
            if not cve_ids and adv_id.startswith("CVE-"):
                cve_ids = [adv_id]

            # Map out a record for each validated CVE identifier associated with the vulnerability
            for cve in (cve_ids if cve_ids else [adv_id]):
                vulnerabilities.append(
                    {
                        "advisory_id": adv_id,
                        "cve_id": cve,
                        "title": title,
                        "url": url,
                        "cvss3_score": cvss_score,
                        "cvss3_vector": cvss_vector,
                    }
                )

        except Exception as e:
            print(f"[Warning] Failed to resolve advisory details for {adv_id}: {e}")
            # Graceful fallback: append basic identifier metadata if the detail call breaks
            vulnerabilities.append(
                {
                    "advisory_id": adv_id,
                    "cve_id": adv_id,
                    "title": "Advisory details unretrievable",
                    "url": "",
                    "cvss3_score": None,
                    "cvss3_vector": "",
                }
            )

    return vulnerabilities


def extract_vulnerabilities_from_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Inspects a crawled dependency DataFrame, unifies all unique
    source/target package versions, scans them for vulnerabilities,
    and returns a structured Vulnerabilities DataFrame.
    """
    if df.empty:
        return pd.DataFrame()

    # 1. Isolate every unique package version pair found across the graph.
    #    Node-only (leaf) records carry a null target, which dropna() removes so
    #    only real package-versions are scanned.
    sources = df[["ecosystem", "source_package", "source_version"]].rename(
        columns={"source_package": "package_name", "source_version": "version"}
    )
    targets = df[["ecosystem", "target_package", "target_version"]].rename(
        columns={"target_package": "package_name", "target_version": "version"}
    )

    unique_elements = pd.concat([sources, targets]).drop_duplicates().dropna()
    vuln_records = []

    print(
        f"\nScanning {len(unique_elements)} unique package-versions for security vulnerabilities..."
    )

    for _, row in tqdm(
        unique_elements.iterrows(), total=len(unique_elements), desc="Fetching CVEs"
    ):
        sys = row["ecosystem"]
        pkg = row["package_name"]
        ver = row["version"]

        cves = get_package_version_vulnerabilities(sys, pkg, ver)
        for cve in cves:
            vuln_records.append(
                {
                    "ecosystem": sys,
                    "package_name": pkg,
                    "version": ver,
                    "advisory_id": cve["advisory_id"],
                    "cve_id": cve["cve_id"],
                    "title": cve["title"],
                    "url": cve["url"],
                    "cvss3_score": cve["cvss3_score"],
                    "cvss3_vector": cve["cvss3_vector"],
                }
            )

    return pd.DataFrame(vuln_records)

In [3]:
# ---------------------------------------------------------------------------
# External enrichment: EPSS, CISA KEV, licenses, and OpenSSF Scorecard
# ---------------------------------------------------------------------------
# These helpers layer real-world context on top of the raw dependency graph:
#   * EPSS  - FIRST.org's probability that a CVE will be exploited in the wild.
#   * KEV   - CISA's catalog of vulnerabilities KNOWN to be actively exploited.
#   * License - the SPDX license each package-version declares (deps.dev).
#   * Scorecard - the OpenSSF repo-health score of the source project (deps.dev).
# Everything routes through the same cached `api_get_with_backoff`, so re-runs
# are cheap. Note: EPSS scores and the KEV catalog change daily; clear the
# matching `.cache` entries (or the cache dir) if you need a fresh pull.


def get_version_metadata(system: str, package_name: str, version: str):
    """Return ``(licenses, project_id, repo_url)`` for one package-version.

    Reuses the very same cached deps.dev GetVersion response that the
    vulnerability scan already fetches, so no extra network cost. ``project_id``
    is deps.dev's canonical source-project key (e.g. ``github.com/foo/bar``),
    taken from the SOURCE_REPO relation when available.
    """
    system = system.upper()
    encoded_package = urllib.parse.quote(package_name, safe="")
    encoded_version = urllib.parse.quote(version, safe="")
    version_url = (
        f"https://api.deps.dev/v3/systems/{system.lower()}/packages/"
        f"{encoded_package}/versions/{encoded_version}"
    )
    try:
        data = api_get_with_backoff(version_url)
    except Exception:
        return [], None, None

    # deps.dev returns SPDX expressions; drop empties. 'non-standard'/'NOASSERTION'
    # are kept verbatim so the graph can still surface "unknown licensing" risk.
    licenses = [lic for lic in (data.get("licenses") or []) if lic]

    project_id, repo_url = None, None
    related = data.get("relatedProjects") or []
    # Prefer the explicit source repository; otherwise take the first related project.
    source = next((p for p in related if p.get("relationType") == "SOURCE_REPO"), None)
    chosen = source or (related[0] if related else None)
    if chosen:
        project_id = chosen.get("projectKey", {}).get("id")
        if project_id:
            repo_url = f"https://{project_id}"

    return licenses, project_id, repo_url


def get_project_scorecard(project_id: str):
    """Fetch deps.dev project metadata + OpenSSF Scorecard for a canonical project
    id (e.g. ``github.com/foo/bar``). Returns a flat dict of repo-health signals,
    or ``None`` if the project can't be resolved."""
    if not project_id:
        return None
    encoded = urllib.parse.quote(project_id, safe="")
    url = f"https://api.deps.dev/v3/projects/{encoded}"
    try:
        data = api_get_with_backoff(url)
    except Exception:
        return None

    def _int(v):
        try:
            return int(v)
        except (TypeError, ValueError):
            return None

    scorecard = data.get("scorecard") or {}
    return {
        "project_id": project_id,
        "repo_url": f"https://{project_id}",
        "stars": _int(data.get("starsCount")),
        "forks": _int(data.get("forksCount")),
        "open_issues": _int(data.get("openIssuesCount")),
        "license": data.get("license"),
        "description": data.get("description"),
        "homepage": data.get("homepage"),
        "scorecard_score": scorecard.get("overallScore"),
        "scorecard_date": scorecard.get("date"),
    }


def extract_package_metadata_from_dataframe(df: pd.DataFrame):
    """For every unique package-version in the crawled graph, pull its declared
    licenses and source-project id from deps.dev, then fetch each unique project's
    OpenSSF Scorecard exactly once.

    Returns ``(licenses_df, projects_df)``:
      * ``licenses_df`` - one row per (package_name, version, spdx_id)
      * ``projects_df`` - one row per (package_name -> source project) with the
        project's scorecard + repo-health signals attached.
    """
    if df.empty:
        return pd.DataFrame(), pd.DataFrame()

    sources = df[["ecosystem", "source_package", "source_version"]].rename(
        columns={"source_package": "package_name", "source_version": "version"}
    )
    targets = df[["ecosystem", "target_package", "target_version"]].rename(
        columns={"target_package": "package_name", "target_version": "version"}
    )
    unique_elements = pd.concat([sources, targets]).drop_duplicates().dropna()

    license_records = []
    pkg_project = {}  # package_name -> source project_id (first non-null wins)

    print(
        f"\nExtracting licenses + source projects for "
        f"{len(unique_elements)} package-versions..."
    )
    for _, row in tqdm(
        unique_elements.iterrows(), total=len(unique_elements), desc="Licenses/projects"
    ):
        sys_, pkg, ver = row["ecosystem"], row["package_name"], row["version"]
        licenses, project_id, _repo = get_version_metadata(sys_, pkg, ver)
        for spdx in licenses:
            license_records.append(
                {"ecosystem": sys_, "package_name": pkg, "version": ver, "spdx_id": spdx}
            )
        if project_id and pkg not in pkg_project:
            pkg_project[pkg] = project_id

    licenses_df = pd.DataFrame(license_records)

    # One scorecard fetch per UNIQUE project, then fan the result back out to every
    # package backed by that project.
    unique_projects = set(pkg_project.values())
    print(f"\nFetching OpenSSF Scorecards for {len(unique_projects)} unique projects...")
    scorecard_cache = {}
    project_records = []
    for pkg, project_id in tqdm(
        pkg_project.items(), total=len(pkg_project), desc="Scorecards"
    ):
        if project_id not in scorecard_cache:
            scorecard_cache[project_id] = get_project_scorecard(project_id)
        sc = scorecard_cache[project_id]
        if sc:
            project_records.append({**sc, "package_name": pkg})

    projects_df = pd.DataFrame(project_records)
    return licenses_df, projects_df


def fetch_epss_scores(cve_ids) -> dict:
    """Batch-fetch EPSS exploitation probabilities from FIRST.org.

    Returns ``{cve_id: {"epss": float, "percentile": float}}``. EPSS only covers
    published CVEs, so GHSA-only advisories simply won't appear in the result.
    """
    cve_ids = sorted({c for c in cve_ids if c and c.startswith("CVE-")})
    scores = {}
    CHUNK = 100  # the EPSS endpoint accepts a comma-separated batch per request
    for i in range(0, len(cve_ids), CHUNK):
        batch = cve_ids[i : i + CHUNK]
        url = "https://api.first.org/data/v1/epss?cve=" + ",".join(batch)
        try:
            data = api_get_with_backoff(url)
        except Exception as e:
            print(f"[Warning] EPSS batch failed: {e}")
            continue
        for row in data.get("data", []):
            cve = row.get("cve")
            if not cve:
                continue
            try:
                scores[cve] = {
                    "epss": float(row["epss"]),
                    "percentile": float(row["percentile"]),
                }
            except (KeyError, TypeError, ValueError):
                continue
    return scores


def fetch_kev_catalog() -> dict:
    """Fetch the CISA Known Exploited Vulnerabilities catalog, keyed by CVE id.

    Presence in this catalog means the vulnerability is being actively exploited
    in the wild - the single strongest "patch this now" signal you can attach.
    """
    url = (
        "https://www.cisa.gov/sites/default/files/feeds/"
        "known_exploited_vulnerabilities.json"
    )
    try:
        data = api_get_with_backoff(url)
    except Exception as e:
        print(f"[Warning] Could not fetch CISA KEV catalog: {e}")
        return {}
    kev = {}
    for v in data.get("vulnerabilities", []):
        cve = v.get("cveID")
        if not cve:
            continue
        kev[cve] = {
            "date_added": v.get("dateAdded"),
            "due_date": v.get("dueDate"),
            "ransomware": v.get("knownRansomwareCampaignUse"),
            "name": v.get("vulnerabilityName"),
        }
    return kev


def enrich_vulnerabilities(vuln_df: pd.DataFrame) -> pd.DataFrame:
    """Augment the vulnerabilities DataFrame with real-world exploitation signals:
    EPSS exploitation probability (FIRST.org) and CISA KEV active-exploitation
    flags. Joined on ``cve_id``; GHSA-only advisories keep null EPSS/KEV fields.
    """
    if vuln_df.empty:
        return vuln_df

    cve_ids = vuln_df["cve_id"].dropna().unique().tolist()
    epss = fetch_epss_scores(cve_ids)
    kev = fetch_kev_catalog()
    print(
        f"EPSS scored {len(epss)} CVEs; "
        f"{sum(c in kev for c in cve_ids)} of this graph's CVEs are in CISA KEV."
    )

    df = vuln_df.copy()
    df["epss_score"] = df["cve_id"].map(lambda c: epss.get(c, {}).get("epss"))
    df["epss_percentile"] = df["cve_id"].map(lambda c: epss.get(c, {}).get("percentile"))
    df["in_kev"] = df["cve_id"].map(lambda c: c in kev)
    df["kev_date_added"] = df["cve_id"].map(lambda c: kev.get(c, {}).get("date_added"))
    df["kev_due_date"] = df["cve_id"].map(lambda c: kev.get(c, {}).get("due_date"))
    df["kev_ransomware"] = df["cve_id"].map(lambda c: kev.get(c, {}).get("ransomware"))
    return df


In [4]:
from IPython.display import display

# Recursively retrieve dependencies for all packages up to the specified depth
final_graph_df = crawl_seed_packages_flat(
    system=SEED_SYSTEM, seed_packages=SEED_PACKAGES, max_depth=SEED_DEPTH
)

print("=" * 50)
print(f"BATCH CRAWL COMPLETE!")
print(
    f"Total Unique Graph Edges Collected: {len(final_graph_df)}, for depth {SEED_DEPTH}"
)
print("=" * 50)

# Preview the final data structure
display(final_graph_df)


=== BFS level 0: crawling 10000 package(s) (discovered total: 10000) ===


Level 0:   0%|          | 0/10000 [00:00<?, ?it/s]

--> Note: Preferred version (1.43.30) has no resolved graph in deps.dev. Fell back to '1.43.27'.
--> Using version '26.2.0' for 'packaging'.
--> Using version '2.7.0' for 'urllib3'.
--> Using version '2026.5.20' for 'certifi'.
--> Using version '3.18.0' for 'idna'.
--> Using version '2.34.2' for 'requests'.
--> Using version '4.15.0' for 'typing-extensions'.
--> Using version '3.4.7' for 'charset-normalizer'.
--> Using version '82.0.1' for 'setuptools'.
--> Note: Preferred version (1.43.30) has no resolved graph in deps.dev. Fell back to '1.43.27'.
--> Note: Preferred version (49.0.0) has no resolved graph in deps.dev. Fell back to '48.0.1'.
--> Using version '3.7.0' for 'aiobotocore'.
--> Using version '2.9.0.post0' for 'python-dateutil'.
--> Using version '1.17.0' for 'six'.
--> Using version '6.0.3' for 'pyyaml'.
--> Using version '2.13.4' for 'pydantic'.
--> Using version '2.20.0' for 'pygments'.
--> Using version '2.0.0' for 'cffi'.
--> Using version '8.4.1' for 'click'.
--> Using

Level 1:   0%|          | 0/313 [00:00<?, ?it/s]

--> Using version '1.12.3' for 'adafruit-circuitpython-typing'.
--> Using version '3.88.0' for 'adafruit-platformdetect'.
--> Using version '1.1.11' for 'adafruit-pureio'.
--> Note: Preferred version (1.16.0) has no resolved graph in deps.dev. Fell back to '1.15.0'.
--> Using version '0.0.3' for 'ag-ui-a2ui-toolkit'.
--> Using version '0.1.14' for 'ai-edge-model-explorer-adapter'.
--> Using version '4.0.3' for 'aim-ui'.
--> Using version '0.5.3.dev8' for 'aimrocks'.
--> Using version '26.0.0' for 'aiogithubapi'.
--> Using version '1.1.2' for 'aiohttp-asgi-connector'.
--> Using version '0.13.3' for 'aiohttp-json-rpc'.
--> Using version '0.10.0' for 'aiopenapi3'.
--> Using version '2.6.0' for 'alibabacloud-actiontrail20200706'.
--> Using version '0.0.2' for 'alibabacloud-apigateway-util'.
--> Using version '6.7.0' for 'alibabacloud-cs20151215'.
--> Using version '0.0.1' for 'alibabacloud-darabonba-time'.
--> Note: Preferred version (7.8.5) has no resolved graph in deps.dev. Fell back to 

Level 2:   0%|          | 0/38 [00:00<?, ?it/s]

--> Using version '5.2.17' for 'adafruit-circuitpython-busdevice'.
--> Using version '4.1.17' for 'adafruit-circuitpython-requests'.
--> Using version '0.1.3' for 'alibabacloud-gateway-pop'.
--> Using version '0.4.2' for 'alibabacloud-gateway-sls'.
--> Using version '0.1.5' for 'choicelib'.
--> Using version '1.0.0' for 'cody-special'.
--> Using version '2.0.0' for 'dacite2'.
--> Note: Preferred version (1.76.17) has no resolved graph in deps.dev. Fell back to '1.76.15'.
--> Using version '2.3.0' for 'fuzzyfinder'.
--> Using version '1.2766.0' for 'gardener-oci'.
--> Using version '0.13.0' for 'gruut-ipa'.
--> Using version '2.0.1' for 'gruut-lang-en'.
--> Using version '0.6.3' for 'honeybee-display'.
--> Using version '0.23.6' for 'honeybee-doe2'.
--> Note: Preferred version (1.121.3) has no resolved graph in deps.dev. Fell back to '1.120.4'.
--> Note: Preferred version (0.5.4) has no resolved graph in deps.dev. Fell back to '0.5.3'.
--> Note: Preferred version (1.66.265) has no resol

,ecosystem,source_package,source_version,target_package,target_version,semver_requirement,graph_depth
0,PYPI,boto3,1.43.27,botocore,1.43.27,"<1.44.0,>=1.43.27",1
1,PYPI,boto3,1.43.27,jmespath,1.1.0,"<2.0.0,>=0.7.1",1
2,PYPI,boto3,1.43.27,s3transfer,0.18.0,"<0.19.0,>=0.18.0",1
3,PYPI,packaging,26.2.0,None,None,,0
4,PYPI,urllib3,2.7.0,None,None,,0
...,...,...,...,...,...,...,...
34340,PYPI,zenodo-client,0.4.2,pydantic,2.13.4,>=2,1
34341,PYPI,zenodo-client,0.4.2,pystow,0.8.16,>=0.4.9,1
34342,PYPI,zenodo-client,0.4.2,requests,2.34.2,,1
34343,PYPI,zenodo-client,0.4.2,tqdm,4.68.2,,1


In [5]:
vulnerabilities_df = extract_vulnerabilities_from_dataframe(final_graph_df)

display(vulnerabilities_df)


Scanning 12050 unique package-versions for security vulnerabilities...


Fetching CVEs:   0%|          | 0/12050 [00:00<?, ?it/s]

[Warning] Failed to resolve advisory details for PYSEC-2025-145: 404 Client Error: Not Found for url: https://api.deps.dev/v3/advisories/PYSEC-2025-145


,ecosystem,package_name,version,advisory_id,cve_id,title,url,cvss3_score,cvss3_vector
0,PYPI,starlette,1.2.1,GHSA-82w8-qh3p-5jfq,CVE-2026-54283,Starlette: request.form() limits silently igno...,https://osv.dev/vulnerability/GHSA-82w8-qh3p-5jfq,7.5,CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:N/I:N/A:H
1,PYPI,starlette,1.2.1,GHSA-jp82-jpqv-5vv3,CVE-2026-54282,Starlette: Unvalidated request path concatenat...,https://osv.dev/vulnerability/GHSA-jp82-jpqv-5vv3,3.7,CVSS:3.1/AV:N/AC:H/PR:N/UI:N/S:U/C:N/I:L/A:N
2,PYPI,sglang,0.5.10.post1,GHSA-36m8-w8qf-g76p,CVE-2026-7304,SGLang: Unauthenticated RCE via --enable-custo...,https://osv.dev/vulnerability/GHSA-36m8-w8qf-g76p,9.8,CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:H
3,PYPI,sglang,0.5.10.post1,GHSA-gwv6-pq6m-p3rq,CVE-2026-7301,SGLanG: Multimodal scheduler deserializes untr...,https://osv.dev/vulnerability/GHSA-gwv6-pq6m-p3rq,9.8,CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:H
4,PYPI,sglang,0.5.10.post1,GHSA-qwrp-wghp-94q2,CVE-2026-7302,SGLang's multimodal generation runtime has an ...,https://osv.dev/vulnerability/GHSA-qwrp-wghp-94q2,9.1,CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:N/I:H/A:H
...,...,...,...,...,...,...,...,...,...
916,PYPI,pillow,12.1.1,GHSA-whj4-6x5x-4v2j,CVE-2026-40192,FITS GZIP decompression bomb in Pillow,https://osv.dev/vulnerability/GHSA-whj4-6x5x-4v2j,7.5,CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:N/I:N/A:H
917,PYPI,pillow,12.1.1,GHSA-wjx4-4jcj-g98j,CVE-2026-42308,Pillow has an integer overflow when processing...,https://osv.dev/vulnerability/GHSA-wjx4-4jcj-g98j,5.5,CVSS:3.1/AV:L/AC:L/PR:L/UI:N/S:U/C:N/I:N/A:H
918,PYPI,pillow,12.1.1,PYSEC-2026-165,CVE-2026-42308,PYSEC-2026-165,https://osv.dev/vulnerability/PYSEC-2026-165,5.5,CVSS:3.1/AV:L/AC:L/PR:L/UI:N/S:U/C:N/I:N/A:H
919,PYPI,wasmtime,24.0.0,PYSEC-2024-311,CVE-2024-47813,PYSEC-2024-311,https://osv.dev/vulnerability/PYSEC-2024-311,2.9,CVSS:3.1/AV:L/AC:H/PR:H/UI:R/S:U/C:N/I:L/A:L


In [6]:
# Enrich the vulnerabilities with real-world exploitation signals (EPSS + CISA KEV).
vulnerabilities_df = enrich_vulnerabilities(vulnerabilities_df)

# Extract per-version licenses and per-project OpenSSF Scorecard / repo-health signals.
licenses_df, projects_df = extract_package_metadata_from_dataframe(final_graph_df)

print(f"\nLicense rows: {len(licenses_df)} | Project rows: {len(projects_df)}")
display(vulnerabilities_df)
display(licenses_df)
display(projects_df)


EPSS scored 354 CVEs; 3 of this graph's CVEs are in CISA KEV.

Extracting licenses + source projects for 12050 package-versions...


Licenses/projects:   0%|          | 0/12050 [00:00<?, ?it/s]


Fetching OpenSSF Scorecards for 7285 unique projects...


Scorecards:   0%|          | 0/9100 [00:00<?, ?it/s]


License rows: 11485 | Project rows: 8851


,ecosystem,package_name,version,advisory_id,cve_id,title,url,cvss3_score,cvss3_vector,epss_score,epss_percentile,in_kev,kev_date_added,kev_due_date,kev_ransomware
0,PYPI,starlette,1.2.1,GHSA-82w8-qh3p-5jfq,CVE-2026-54283,Starlette: request.form() limits silently igno...,https://osv.dev/vulnerability/GHSA-82w8-qh3p-5jfq,7.5,CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:N/I:N/A:H,NaN,NaN,False,NaN,NaN,NaN
1,PYPI,starlette,1.2.1,GHSA-jp82-jpqv-5vv3,CVE-2026-54282,Starlette: Unvalidated request path concatenat...,https://osv.dev/vulnerability/GHSA-jp82-jpqv-5vv3,3.7,CVSS:3.1/AV:N/AC:H/PR:N/UI:N/S:U/C:N/I:L/A:N,NaN,NaN,False,NaN,NaN,NaN
2,PYPI,sglang,0.5.10.post1,GHSA-36m8-w8qf-g76p,CVE-2026-7304,SGLang: Unauthenticated RCE via --enable-custo...,https://osv.dev/vulnerability/GHSA-36m8-w8qf-g76p,9.8,CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:H,0.00585,0.43330,False,NaN,NaN,NaN
3,PYPI,sglang,0.5.10.post1,GHSA-gwv6-pq6m-p3rq,CVE-2026-7301,SGLanG: Multimodal scheduler deserializes untr...,https://osv.dev/vulnerability/GHSA-gwv6-pq6m-p3rq,9.8,CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:H,0.00399,0.31633,False,NaN,NaN,NaN
4,PYPI,sglang,0.5.10.post1,GHSA-qwrp-wghp-94q2,CVE-2026-7302,SGLang's multimodal generation runtime has an ...,https://osv.dev/vulnerability/GHSA-qwrp-wghp-94q2,9.1,CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:N/I:H/A:H,0.00386,0.30280,False,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
916,PYPI,pillow,12.1.1,GHSA-whj4-6x5x-4v2j,CVE-2026-40192,FITS GZIP decompression bomb in Pillow,https://osv.dev/vulnerability/GHSA-whj4-6x5x-4v2j,7.5,CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:N/I:N/A:H,0.00485,0.37922,False,NaN,NaN,NaN
917,PYPI,pillow,12.1.1,GHSA-wjx4-4jcj-g98j,CVE-2026-42308,Pillow has an integer overflow when processing...,https://osv.dev/vulnerability/GHSA-wjx4-4jcj-g98j,5.5,CVSS:3.1/AV:L/AC:L/PR:L/UI:N/S:U/C:N/I:N/A:H,0.00114,0.01729,False,NaN,NaN,NaN
918,PYPI,pillow,12.1.1,PYSEC-2026-165,CVE-2026-42308,PYSEC-2026-165,https://osv.dev/vulnerability/PYSEC-2026-165,5.5,CVSS:3.1/AV:L/AC:L/PR:L/UI:N/S:U/C:N/I:N/A:H,0.00114,0.01729,False,NaN,NaN,NaN
919,PYPI,wasmtime,24.0.0,PYSEC-2024-311,CVE-2024-47813,PYSEC-2024-311,https://osv.dev/vulnerability/PYSEC-2024-311,2.9,CVSS:3.1/AV:L/AC:H/PR:H/UI:R/S:U/C:N/I:L/A:L,0.00152,0.04665,False,NaN,NaN,NaN


,ecosystem,package_name,version,spdx_id
0,PYPI,boto3,1.43.27,Apache-2.0
1,PYPI,packaging,26.2.0,Apache-2.0 OR BSD-2-Clause
2,PYPI,urllib3,2.7.0,MIT
3,PYPI,certifi,2026.5.20,MPL-2.0
4,PYPI,idna,3.18.0,BSD-3-Clause
...,...,...,...,...
11480,PYPI,invenio-theme,4.8.0,MIT
11481,PYPI,simplekv,0.14.1,MIT
11482,PYPI,pytz,2024.1.0,MIT
11483,PYPI,ladybug-comfort,0.18.117,AGPL-3.0


,project_id,repo_url,stars,forks,open_issues,license,description,homepage,scorecard_score,scorecard_date,package_name
0,github.com/boto/boto3,https://github.com/boto/boto3,9833,1973,188,Apache-2.0,"Boto3, an AWS SDK for Python",https://aws.amazon.com/sdk-for-python/,7.5,2026-06-08T00:00:00Z,boto3
1,github.com/pypa/packaging,https://github.com/pypa/packaging,732,300,113,non-standard,Core utilities for Python packages,https://packaging.pypa.io/,8.9,2026-06-08T00:00:00Z,packaging
2,github.com/urllib3/urllib3,https://github.com/urllib3/urllib3,4025,1392,218,MIT,urllib3 is a user-friendly HTTP client library...,https://urllib3.readthedocs.io,8.5,2026-06-08T00:00:00Z,urllib3
3,github.com/certifi/python-certifi,https://github.com/certifi/python-certifi,977,289,2,non-standard,(Python Distribution) A carefully curated coll...,,5.9,2026-06-08T00:00:00Z,certifi
4,github.com/kjd/idna,https://github.com/kjd/idna,287,126,0,BSD-3-Clause,Internationalized Domain Names for Python (IDN...,,8.6,2026-06-08T00:00:00Z,idna
...,...,...,...,...,...,...,...,...,...,...,...
8846,github.com/inveniosoftware/invenio-theme,https://github.com/inveniosoftware/invenio-theme,9,62,17,MIT,Invenio standard theme.,https://invenio-theme.readthedocs.io,4.3,2026-06-08T00:00:00Z,invenio-theme
8847,github.com/rr2do2/maxminddb-geolite2,https://github.com/rr2do2/maxminddb-geolite2,63,11,4,non-standard,MaxMind GeoLite2 database as a convenient Pyth...,,1.9,2026-06-08T00:00:00Z,maxminddb-geolite2
8848,github.com/mbr/simplekv,https://github.com/mbr/simplekv,159,48,6,MIT,A simple key-value store for binary data.,http://simplekv.readthedocs.io,3.0,2026-06-08T00:00:00Z,simplekv
8849,github.com/ladybug-tools/ladybug-comfort,https://github.com/ladybug-tools/ladybug-comfort,17,21,12,AGPL-3.0,🐞 :tired_face: :smile: :sweat: Ladybug extensi...,https://www.ladybug.tools/ladybug-comfort/docs/,4.4,2026-06-08T00:00:00Z,ladybug-comfort


## The graph schema

Our graph schema will be as follows:

`(:Package)-[:DEPENDS_ON]->(:Package)`
`(:Package)-[:HAS_VULNERABILITY]->(:Vulnerability)`
`(:Package)-[:HAS_LICENSE]->(:License)`
`(:Package)-[:PUBLISHED_FROM]->(:Project)`

`:Package` nodes will have the following properties:

- `name`: The name of the package.

`:Vulnerability` nodes will have the following properties:

- `advisory_id`: The unique identifier for the vulnerability.
- `cve_id`: The Common Vulnerabilities and Exposures (CVE) identifier for the vulnerability.
- `title`: The title or brief description of the vulnerability.
- `url`: The URL with more information about the vulnerability.
- `cvss3_score`: The CVSS v3 score indicating the severity of the vulnerability.
- `cvss3_vector`: The CVSS v3 vector string representing the vulnerability's characteristics.
- `epss_score`: The [EPSS](https://www.first.org/epss/) probability (0-1) that this CVE will be exploited in the wild within the next 30 days.
- `epss_percentile`: The CVE's EPSS percentile rank relative to all scored CVEs.
- `in_kev`: Boolean flag - is this CVE in [CISA's Known Exploited Vulnerabilities](https://www.cisa.gov/known-exploited-vulnerabilities-catalog) catalog (i.e. actively exploited)?
- `kev_date_added`: The date CISA added the CVE to the KEV catalog (if applicable).
- `kev_due_date`: The CISA-mandated remediation due date (if applicable).
- `kev_ransomware`: Whether the CVE is known to be used in ransomware campaigns.

`:License` nodes (one per distinct [SPDX](https://spdx.org/licenses/) license) will have:

- `spdx_id`: The SPDX license identifier (e.g. `MIT`, `Apache-2.0`, or `non-standard`).

`:Project` nodes (the source repository a package is published from, enriched with the [OpenSSF Scorecard](https://securityscorecards.dev/)) will have:

- `id`: The canonical deps.dev project key (e.g. `github.com/psf/requests`).
- `repo_url`: The repository URL.
- `stars`, `forks`, `open_issues`: Repository popularity / activity signals.
- `license`: The license deps.dev reports for the repository.
- `description`, `homepage`: Descriptive metadata.
- `scorecard_score`: The overall OpenSSF Scorecard score (0-10) of repo security posture.
- `scorecard_date`: The date the scorecard was computed.

The `:DEPENDS_ON` relationship will have the following properties:

- `semver_requirement`: The semantic versioning requirement for the dependency.
- `source_version`: The version of the source package.
- `target_version`: The version of the target package.

The `:HAS_VULNERABILITY` relationship will have:

- `version`: The version of the package affected by the vulnerability.

The `:HAS_LICENSE` relationship will have:

- `version`: The package version that declared this license.


In [7]:
from neo4j import GraphDatabase


def create_constraints(tx):
    """Creates uniqueness constraints for the graph."""
    # Note: 'CREATE CONSTRAINT IF NOT EXISTS' is standard for Neo4j 4.4+
    tx.run(
        "CREATE CONSTRAINT package_name_unique IF NOT EXISTS FOR (p:Package) REQUIRE p.name IS UNIQUE"
    )
    tx.run(
        "CREATE CONSTRAINT vulnerability_id_unique IF NOT EXISTS FOR (v:Vulnerability) REQUIRE v.advisory_id IS UNIQUE"
    )
    tx.run(
        "CREATE CONSTRAINT license_spdx_unique IF NOT EXISTS FOR (l:License) REQUIRE l.spdx_id IS UNIQUE"
    )
    tx.run(
        "CREATE CONSTRAINT project_id_unique IF NOT EXISTS FOR (proj:Project) REQUIRE proj.id IS UNIQUE"
    )


def load_package_dependencies(tx, records):
    """
    Loads package dependency records into Neo4j safely by isolating
    node merges to distinct sets before drawing relationships.

    Leaf packages arrive as node-only records with a null target_package: their
    source node is still merged, but the null target is filtered out so no spurious
    Package node or DEPENDS_ON edge is created.
    """
    # Step 1: Merge all UNIQUE source packages in this batch
    tx.run(
        """
    UNWIND $records AS record
    WITH DISTINCT record.source_package AS pkg_name
    WHERE pkg_name IS NOT NULL
    MERGE (:Package {name: pkg_name})
    """,
        records=records,
    )

    # Step 2: Merge all UNIQUE target packages in this batch (skip null/leaf targets)
    tx.run(
        """
    UNWIND $records AS record
    WITH DISTINCT record.target_package AS pkg_name
    WHERE pkg_name IS NOT NULL
    MERGE (:Package {name: pkg_name})
    """,
        records=records,
    )

    # Step 3: Match the now-guaranteed nodes and safely merge the relationship.
    #         Records with a null target (leaf packages) are skipped here.
    query = """
    UNWIND $records AS record
    WITH record
    WHERE record.target_package IS NOT NULL
    MATCH (source:Package {name: record.source_package})
    MATCH (target:Package {name: record.target_package})
    MERGE (source)-[r:DEPENDS_ON {
        source_version: record.source_version,
        target_version: record.target_version
    }]->(target)
    SET r.semver_requirement = record.semver_requirement
    """
    tx.run(query, records=records)


def load_vulnerabilities(tx, records):
    """
    Loads vulnerability records safely. Since one CVE can affect multiple
    packages/versions, this prevents duplicate Vulnerability node collisions.

    Beyond CVSS severity, each vulnerability now carries real-world exploitation
    signals: EPSS exploit probability and CISA KEV active-exploitation flags.
    """
    # Step 1: Ensure all packages in this vulnerability batch exist
    tx.run(
        """
    UNWIND $records AS record
    WITH DISTINCT record.package_name AS pkg_name
    MERGE (:Package {name: pkg_name})
    """,
        records=records,
    )

    # Step 2: Ensure all unique vulnerabilities in this batch are merged cleanly
    tx.run(
        """
    UNWIND $records AS record
    WITH DISTINCT record.advisory_id AS adv_id
    MERGE (:Vulnerability {advisory_id: adv_id})
    """,
        records=records,
    )

    # Step 3: Map properties and draw the edges using safe MATCH lookups
    query = """
    UNWIND $records AS record
    MATCH (source:Package {name: record.package_name})
    MATCH (v:Vulnerability {advisory_id: record.advisory_id})
    MERGE (source)-[r:HAS_VULNERABILITY]->(v)
    SET v.title = record.title,
        v.cve_id = record.cve_id,
        v.url = record.url,
        v.cvss3_score = record.cvss3_score,
        v.cvss3_vector = record.cvss3_vector,
        v.epss_score = record.epss_score,
        v.epss_percentile = record.epss_percentile,
        v.in_kev = record.in_kev,
        v.kev_date_added = record.kev_date_added,
        v.kev_due_date = record.kev_due_date,
        v.kev_ransomware = record.kev_ransomware
    SET r.version = record.version
    """
    tx.run(query, records=records)


def load_licenses(tx, records):
    """Loads per-version license declarations. A package-version may declare more
    than one license, so HAS_LICENSE is keyed by version to keep those distinct."""
    # Step 1: Merge the (deduplicated) License nodes
    tx.run(
        """
    UNWIND $records AS record
    WITH DISTINCT record.spdx_id AS spdx
    WHERE spdx IS NOT NULL
    MERGE (:License {spdx_id: spdx})
    """,
        records=records,
    )

    # Step 2: Link each package to the licenses its versions declare
    tx.run(
        """
    UNWIND $records AS record
    WITH record WHERE record.spdx_id IS NOT NULL
    MATCH (p:Package {name: record.package_name})
    MATCH (l:License {spdx_id: record.spdx_id})
    MERGE (p)-[rel:HAS_LICENSE {version: record.version}]->(l)
    """,
        records=records,
    )


def load_projects(tx, records):
    """Loads source-project nodes enriched with OpenSSF Scorecard + repo-health
    signals, and links each package to the project it is published from."""
    # Step 1: Merge Project nodes and set their health signals
    tx.run(
        """
    UNWIND $records AS record
    WITH record WHERE record.project_id IS NOT NULL
    MERGE (proj:Project {id: record.project_id})
    SET proj.repo_url        = record.repo_url,
        proj.stars           = record.stars,
        proj.forks           = record.forks,
        proj.open_issues     = record.open_issues,
        proj.license         = record.license,
        proj.description     = record.description,
        proj.homepage        = record.homepage,
        proj.scorecard_score = record.scorecard_score,
        proj.scorecard_date  = record.scorecard_date
    """,
        records=records,
    )

    # Step 2: Connect each package to its source project
    tx.run(
        """
    UNWIND $records AS record
    WITH record WHERE record.project_id IS NOT NULL
    MATCH (p:Package {name: record.package_name})
    MATCH (proj:Project {id: record.project_id})
    MERGE (p)-[:PUBLISHED_FROM]->(proj)
    """,
        records=records,
    )


def to_clean_records(frame):
    """Convert a DataFrame to driver-ready dict records, coercing pandas NaN to
    None so missing values reach Cypher as genuine nulls (keeping IS NOT NULL
    filters and optional properties reliable)."""
    if frame is None or frame.empty:
        return []
    return frame.astype(object).where(frame.notna(), None).to_dict("records")


# Initialize Driver
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

try:
    # Setup Schema (Constraints)
    with driver.session(database=NEO4J_DATABASE) as session:
        session.execute_write(create_constraints)
        print("[Success] Uniqueness constraints verified/created.")

    batch_size = 1000

    # Batch Load Dependency edges.
    # Coerce any pandas NaN to None so null targets (leaf packages) reach Cypher as
    # genuine nulls, keeping the `IS NOT NULL` filters reliable.
    package_records = to_clean_records(final_graph_df)
    for i in range(0, len(package_records), batch_size):
        batch = package_records[i : i + batch_size]
        with driver.session(database=NEO4J_DATABASE) as session:
            session.execute_write(load_package_dependencies, batch)
        print(
            f"[Success] Loaded batch {i // batch_size + 1} ({len(batch)} package records)."
        )

    # Load vulnerabilities (now carrying EPSS + KEV enrichment)
    vulnerability_records = to_clean_records(vulnerabilities_df)
    for i in range(0, len(vulnerability_records), batch_size):
        batch = vulnerability_records[i : i + batch_size]
        with driver.session(database=NEO4J_DATABASE) as session:
            session.execute_write(load_vulnerabilities, batch)
        print(
            f"[Success] Loaded vulnerability batch {i // batch_size + 1} ({len(batch)} vulnerability records)."
        )

    # Load license declarations
    license_records = to_clean_records(licenses_df)
    for i in range(0, len(license_records), batch_size):
        batch = license_records[i : i + batch_size]
        with driver.session(database=NEO4J_DATABASE) as session:
            session.execute_write(load_licenses, batch)
        print(
            f"[Success] Loaded license batch {i // batch_size + 1} ({len(batch)} license records)."
        )

    # Load source projects + OpenSSF Scorecard signals
    project_records = to_clean_records(projects_df)
    for i in range(0, len(project_records), batch_size):
        batch = project_records[i : i + batch_size]
        with driver.session(database=NEO4J_DATABASE) as session:
            session.execute_write(load_projects, batch)
        print(
            f"[Success] Loaded project batch {i // batch_size + 1} ({len(batch)} project records)."
        )

    print("\n" + "=" * 50)
    print("GRAPH LOADING COMPLETE!")
    print("=" * 50)

finally:
    driver.close()


[Success] Uniqueness constraints verified/created.
[Success] Loaded batch 1 (1000 package records).
[Success] Loaded batch 2 (1000 package records).
[Success] Loaded batch 3 (1000 package records).
[Success] Loaded batch 4 (1000 package records).
[Success] Loaded batch 5 (1000 package records).
[Success] Loaded batch 6 (1000 package records).
[Success] Loaded batch 7 (1000 package records).
[Success] Loaded batch 8 (1000 package records).
[Success] Loaded batch 9 (1000 package records).
[Success] Loaded batch 10 (1000 package records).
[Success] Loaded batch 11 (1000 package records).
[Success] Loaded batch 12 (1000 package records).
[Success] Loaded batch 13 (1000 package records).
[Success] Loaded batch 14 (1000 package records).
[Success] Loaded batch 15 (1000 package records).
[Success] Loaded batch 16 (1000 package records).
[Success] Loaded batch 17 (1000 package records).
[Success] Loaded batch 18 (1000 package records).
[Success] Loaded batch 19 (1000 package records).
[Success